# Clasificación de Mensajes: Detección de Spam con Regresión Logística

En esta práctica, implementaremos un modelo de aprendizaje automático capaz de distinguir entre correos electrónicos legítimos (HAM) y correos no deseados (SPAM). Utilizaremos el conjunto de datos TREC 2007, disponible en Kaggle.

El núcleo del proyecto es la **Regresión Logística**. A diferencia de otros modelos predictivos, este utiliza la función sigmoide para transformar una combinación lineal de variables en un valor de probabilidad:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \quad \text{donde} \quad z = \beta_0 + \beta_1 x_1 + ... + \beta_n x_n$$

Este valor resultante (entre 0 y 1) nos indica la confianza del modelo en que un mensaje pertenezca a la categoría de SPAM.

Puedes consultar la fuente de datos original en [este enlace](https://www.kaggle.com/datasets/imdeepmind/preprocessed-trec-2007-public-corpus-dataset).

### 0. Preparación del Entorno
Comenzamos importando las herramientas de análisis de datos, procesamiento de texto y modelado estadístico.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

%matplotlib inline

### 1. Lectura de la Información
Cargamos el archivo CSV que contiene la base de datos de correos preprocesados.

In [2]:
df = pd.read_csv("processed_data.csv")
df.head()

,label,subject,email_to,email_from,message
0,1,"Generic Cialis, branded quality@",the00@speedy.uwaterloo.ca,"""Tomas Jacobs"" <RickyAmes@aol.com>",Content-Type: text/html;\nContent-Transfer-Enc...
1,0,Typo in /debian/README,debian-mirrors@lists.debian.org,Yan Morin <yan.morin@savoirfairelinux.com>,"Hi, i've just updated from the gulus and I che..."
2,1,authentic viagra,<the00@plg.uwaterloo.ca>,"""Sheila Crenshaw"" <7stocknews@tractionmarketin...","Content-Type: text/plain;\n\tcharset=""iso-8859..."
3,1,Nice talking with ya,opt4@speedy.uwaterloo.ca,"""Stormy Dempsey"" <vqucsmdfgvsg@ruraltek.com>","Hey Billy, \n\nit was really fun going out the..."
4,1,or trembling; stomach cramps; trouble in sleep...,ktwarwic@speedy.uwaterloo.ca,"""Christi T. Jernigan"" <dcube@totalink.net>",Content-Type: multipart/alternative;\n ...


Revisamos la distribución de las etiquetas para entender el balance del dataset:

In [3]:
df['label'].value_counts()

label
1    50199
0    25220
Name: count, dtype: int64

Referencia de etiquetas:
* **0**: Correo legítimo (HAM)
* **1**: Correo no deseado (SPAM)

### 2. Depuración y Limpieza del Dataset
Para garantizar la calidad del modelo, identificamos y eliminamos registros que puedan inducir a error, como duplicados o valores nulos.

In [4]:
# Identificación de registros idénticos
total_duplicados = df.duplicated(subset=['subject', 'email_from', 'email_to', 'message']).sum()
print(f"Registros duplicados encontrados: {total_duplicados}")

Registros duplicados encontrados: 1513


In [5]:
# Verificación de campos vacíos en columnas críticas
df[['subject', 'message', 'label']].isnull().sum()

subject     793
message    1487
label         0
dtype: int64

In [6]:
# Procedemos a eliminar las filas duplicadas
df = df.drop_duplicates(subset=['subject', 'email_from', 'email_to', 'message']).reset_index(drop=True)
print(f"Nuevo tamaño del dataset: {df.shape}")

Nuevo tamaño del dataset: (73906, 5)


### 3. Transformación de Texto a Datos Numéricos
Dado que los modelos matemáticos no procesan texto crudo, debemos convertir las palabras en representaciones numéricas. Utilizaremos la técnica de 'Bolsa de Palabras' (Bag of Words) mediante `CountVectorizer`.

In [7]:
# Consolidamos el asunto y el mensaje en una sola variable textual
df['text'] = df['subject'].fillna('') + ' ' + df['message'].fillna('')

In [8]:
# División cronológica: 80% para entrenamiento y 20% para validación
punto_corte = int(len(df) * 0.8)

X_train_raw = df['text'][:punto_corte]
X_test_raw  = df['text'][punto_corte:]
y_train = df['label'][:punto_corte]
y_test  = df['label'][punto_corte:]

In [9]:
# Configuramos el vectorizador para considerar las 10,000 palabras más frecuentes
vectorizer = CountVectorizer(max_features=10000)

# Ajustamos el vocabulario con los datos de entrenamiento y transformamos ambos conjuntos
X_train = vectorizer.fit_transform(X_train_raw)
X_test  = vectorizer.transform(X_test_raw)

print(f"Matriz de entrenamiento generada: {X_train.shape}")

Matriz de entrenamiento generada: (59124, 10000)


### 4. Ajuste del Modelo Predictivo
Entrenamos nuestro clasificador de Regresión Logística. Aumentamos el número de iteraciones para asegurar que el algoritmo converja correctamente con el volumen de datos.

In [10]:
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train, y_train)

print(f"Total de coeficientes aprendidos: {modelo.coef_.shape[1]}")

Total de coeficientes aprendidos: 10000


Cada término del vocabulario recibe un peso (coeficiente). Un peso positivo sugiere que la palabra está ligada al SPAM, mientras que uno negativo indica una mayor probabilidad de ser un correo legítimo.

In [11]:
# Análisis de las palabras con mayor peso en la clasificación
nombres_palabras = vectorizer.get_feature_names_out()
pesos = modelo.coef_[0]

ranking_spam = sorted(zip(pesos, nombres_palabras), reverse=True)[:15]
ranking_ham  = sorted(zip(pesos, nombres_palabras))[:15]

print("Palabras clave para detectar SPAM:")
for p, palabra in ranking_spam:
    print(f"  {palabra:<20} Coef = {p:.3f}")

print("\nPalabras clave para detectar HAM:")
for p, palabra in ranking_ham:
    print(f"  {palabra:<20} Coef = {p:.3f}")

Palabras clave para detectar SPAM:
  medhelp              Coef = 2.745
  gif                  Coef = 1.263
  hk                   Coef = 1.117
  penis                Coef = 1.099
  girl                 Coef = 0.958
  adf                  Coef = 0.932
  producttestpanel     Coef = 0.914
  type                 Coef = 0.910
  id                   Coef = 0.856
  viewing              Coef = 0.850
  jp                   Coef = 0.825
  rent                 Coef = 0.823
  sort                 Coef = 0.814
  alt                  Coef = 0.811
  look                 Coef = 0.802

Palabras clave para detectar HAM:
  lh                   Coef = -2.854
  wrote                Coef = -2.462
  perl                 Coef = -2.405
  reform               Coef = -1.990
  following            Coef = -1.567
  debian               Coef = -1.531
  samba                Coef = -1.514
  hello                Coef = -1.456
  function             Coef = -1.451
  lists                Coef = -1.389
  matrix            

### 5. Validación y Resultados Finales
Evaluamos el desempeño del modelo utilizando el conjunto de datos de prueba que el sistema no ha visto anteriormente.

In [12]:
# Realizamos las predicciones de clase y obtenemos las probabilidades asociadas
predicciones = modelo.predict(X_test)
probabilidades = modelo.predict_proba(X_test)

print(f"{'Predicción':<12} {'Prob(HAM)':<10} {'Prob(SPAM)':<10} {'Real'}") 
print("-" * 50)
for i in range(5):
    clase_pred = "SPAM" if predicciones[i] == 1 else "HAM"
    clase_real = "SPAM" if list(y_test)[i] == 1 else "HAM"
    print(f"{clase_pred:<12} {probabilidades[i][0]:<10.3f} {probabilidades[i][1]:<10.3f} {clase_real}")

Predicción   Prob(HAM)  Prob(SPAM) Real
--------------------------------------------------
SPAM         0.000      1.000      SPAM
SPAM         0.000      1.000      SPAM
SPAM         0.000      1.000      SPAM
HAM          1.000      0.000      HAM
SPAM         0.000      1.000      SPAM


In [13]:
# Cálculo de la precisión global del modelo
precision = accuracy_score(y_test, predicciones)
print(f"Exactitud (Accuracy) lograda: {precision:.3f}")

Exactitud (Accuracy) lograda: 0.991
